# HW2: multiple object video detection

**How to run.** The notebook runs on Google Colab or on a local machine with an NVIDIA GPU. It detects Colab by itself.
- **Colab:** results go to `MyDrive/ai-ucu-hw-2` on Google Drive.
- **Local:** results go to `outputs/` next to the notebook, and the prepared training folder to `prep/`. Install a PyTorch build for your GPU first (an RTX 50xx needs CUDA 12.8 or newer), then the rest:
  ```
  pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128
  pip install -r requirements.txt
  ```
  The data is taken from `multiple-object-video-detection-hw-2-ucu-ai-2026/` if it is there, otherwise downloaded with `kagglehub`. The Kaggle token is read from `KAGGLE_API_TOKEN`, from `.env` (`KAGGLE_API_TOKEN=KGAT_...`) or asked for.

Flags in the first code cell: `SMOKE_TEST` runs every step on a few frames and 1 epoch to catch errors (it never submits); `TRAIN` trains models that have no saved weights yet; finished runs are read from `experiment_log.csv` and not repeated unless `FORCE` or `ONLY` says so.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
COMPETITION = "multiple-object-video-detection-hw-2-ucu-ai-2026"
DRIVE = "/content/drive/MyDrive/ai-ucu-hw-2" if IN_COLAB else os.path.abspath("outputs")
PREP_ARCHIVE = f"{DRIVE}/prep.tar"
SMOKE_TEST = False
RUNS = f"{DRIVE}/runs-smoke" if SMOKE_TEST else f"{DRIVE}/runs"
TRAIN = True
FORCE = False
ONLY = []
SELECT_METRIC = 'mAP_testlike'  # val mAP on the clips whose resolution occurs in test
TRY_P2_HEAD = False  # experiment 8, the slowest training run
USE_TRAINVAL = True  # retrain the final YOLO on train + val and submit it as a second file
RESUBMIT = ''  # a saved submission file in RUNS to send once, e.g. 'submission-yolo11n-1280-full-best-1280.csv'
SUBMIT = True
STOP_RUNTIME = True

: 

In [ ]:
import os
import shutil
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !nvidia-smi --query-gpu=name,memory.total --format=csv || echo "CPU runtime"
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(RUNS, exist_ok=True)

%pip install -q -U kagglehub ultralytics faster-coco-eval kaggle six ensemble-boxes seaborn scikit-learn

free_gb = shutil.disk_usage("/content" if IN_COLAB else ".").free / 1e9
print(f"Free disk: {free_gb:.0f} GB")
if IN_COLAB:
    print(f"Free on Drive: {shutil.disk_usage(DRIVE).free / 1e9:.1f} GB")


def drive_ok():
    return not IN_COLAB or os.path.ismount("/content/drive")


def stop_runtime():
    if IN_COLAB and STOP_RUNTIME:
        if not drive_ok():
            print("Drive is not mounted, the runtime stays on so the results can be copied")
            return
        from google.colab import runtime
        drive.flush_and_unmount(timeout_ms=600_000)
        runtime.unassign()


def stop_on_error(shell, etype, value, tb, tb_offset=None):
    shell.showtraceback((etype, value, tb), tb_offset=tb_offset)
    if IN_COLAB and STOP_RUNTIME:
        import traceback
        with open(f"{RUNS}/last_error.txt", "w") as f:
            f.write("".join(traceback.format_exception(etype, value, tb)))
    stop_runtime()


get_ipython().set_custom_exc((Exception,), stop_on_error)

In [ ]:
import contextlib
import gc
import io
import json
import math
import re
import time
import warnings
from functools import partial
from getpass import getpass
from pathlib import Path

os.environ['YOLO_VERBOSE'] = 'False'

import cv2
import faster_coco_eval
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torchvision
from ensemble_boxes import weighted_boxes_fusion
from PIL import Image
from sklearn.model_selection import GroupKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.io import read_image
from torchvision.models.detection import (fasterrcnn_resnet50_fpn_v2, retinanet, retinanet_resnet50_fpn_v2,
                                          roi_heads)
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torchvision.ops import batched_nms, box_iou, generalized_box_iou_loss
from tqdm.auto import tqdm
from ultralytics import RTDETR, YOLO
from ultralytics.utils import LOGGER

faster_coco_eval.init_as_pycocotools()
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

warnings.filterwarnings('ignore', message='grid_sampler_2d_backward_cuda')
LOGGER.setLevel('WARNING')

if torch.cuda.is_available():
    arch = 'sm_{}{}'.format(*torch.cuda.get_device_capability())
    print(torch.cuda.get_device_name(), arch, '| torch', torch.__version__, '| CUDA', torch.version.cuda)
    assert arch in torch.cuda.get_arch_list(), f'This PyTorch build has no kernels for {arch}, install a newer CUDA build'

CLASSES = ['person', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'bus', 'motor']
COLORS = {'train': '#2a78d6', 'val': '#eb6834', 'test': '#1baf7a'}


def draw_boxes(image, boxes, color):
    thickness = max(2, image.shape[1] // 700)
    for x, y, w, h in boxes[['x', 'y', 'w', 'h']].to_numpy(int):
        cv2.rectangle(image, (x, y), (x + w, y + h), color, thickness)
    return image

In [ ]:
from getpass import getpass


def load_kaggle_token():
    if os.environ.get("KAGGLE_API_TOKEN"):
        return "environment"
    if IN_COLAB:
        from google.colab import userdata
        try:
            os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
            return "Colab Secrets"
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException) as e:
            print(f"Colab secret unavailable ({type(e).__name__}), asking instead.")
    else:
        for env_file in (".env", "../hw-1/.env"):
            for line in open(env_file) if os.path.exists(env_file) else []:
                if line.startswith("KAGGLE_API_TOKEN="):
                    os.environ["KAGGLE_API_TOKEN"] = line.split("=", 1)[1].strip().strip("\"'")
                    return env_file
    os.environ["KAGGLE_API_TOKEN"] = getpass("Kaggle API token (KGAT_...): ")
    return "prompt"


print("Kaggle token loaded from:", load_kaggle_token())

In [ ]:
LOCAL_DATA = Path(COMPETITION)
if (LOCAL_DATA / 'test_images.csv').exists():
    path = str(LOCAL_DATA.resolve())
else:
    path = kagglehub.competition_download(COMPETITION)

print("Path to competition files:", path)

### Submit a saved file
Sends the submission named in `RESUBMIT` if it was never sent to Kaggle. It needs only the cells above. The file is taken from `RUNS`, rebuilt from the per-clip predictions saved there, or, if neither exists, predicted again from the saved weights its name points to (`<weights>[-best]-<imgsz>[-tta]`, score threshold 0.01). It is skipped if a submission with the same message already exists.

In [ ]:
def predict_again(file):
    weights, best, imgsz, tta = re.fullmatch(r'(.+?)(-best)?-(\d+)(-tta)?', file.stem.removeprefix('submission-')).groups()
    checkpoint = Path(RUNS) / weights / 'weights' / ('best.pt' if best else 'last.pt')
    if not checkpoint.exists():
        print('No weights at', checkpoint)
        return None
    data = Path(path)
    test = pd.read_csv(data / 'test_images.csv')
    results = YOLO(str(checkpoint)).predict([str(data / f) for f in test.file_name], stream=True, imgsz=int(imgsz),
                                            conf=0.01, max_det=500, augment=bool(tta), verbose=False)
    strings = []
    for result in tqdm(results, total=len(test), desc='test'):
        rows = zip(result.boxes.xyxy.tolist(), result.boxes.conf.tolist(), result.boxes.cls.int().tolist())
        strings.append(' '.join(f'{c} {s:.4f} {x1:.1f} {y1:.1f} {x2 - x1:.1f} {y2 - y1:.1f}'
                                for (x1, y1, x2, y2), s, c in rows))
    pd.DataFrame({'image_id': test.image_id, 'PredictionString': strings}).to_csv(file, index=False)
    return file


def saved_submission(file):
    if file.exists():
        return file
    parts = sorted((Path(RUNS) / 'predictions' / file.stem.removeprefix('submission-')).glob('*.json'))
    predictions = {k: v for part in parts for k, v in json.loads(part.read_text()).items()}
    if len(parts) == 48 and len(predictions) == 3811:
        pd.DataFrame(sorted(predictions.items()), columns=['image_id', 'PredictionString']).to_csv(file, index=False)
        return file
    return predict_again(file)


if RESUBMIT:
    message = Path(RESUBMIT).stem.removeprefix('submission-')
    submitted = !kaggle competitions submissions -c {COMPETITION} -v
    if any(f',{message},' in line for line in submitted):
        print('Already submitted:', message)
    elif (file := saved_submission(Path(RUNS) / RESUBMIT)) is not None:
        !kaggle competitions submit -c {COMPETITION} -f "{file}" -m "{message}"
        !kaggle competitions submissions -c {COMPETITION}

## Data overview
Drone video clips from VisDrone with 8 object classes. Train and val have labels, test has only frames. This section looks at the data, finds what is wrong with it, and puts the fixes into one `clean()`.

In [ ]:
DATA = Path(path)


def load_split(split):
    coco = json.loads((DATA / split / 'annotations_coco.json').read_text())
    images = pd.DataFrame(coco['images']).rename(columns={'id': 'image_id'})
    boxes = pd.DataFrame(coco['annotations'])
    boxes[['x', 'y', 'w', 'h']] = np.array(boxes['bbox'].tolist())
    boxes = boxes.drop(columns='bbox').merge(
        images[['image_id', 'video', 'frame_id', 'width', 'height']], on='image_id')
    return images, boxes


train_images, train_boxes = load_split('train')
val_images, val_boxes = load_split('val')
test_images = pd.read_csv(DATA / 'test_images.csv')

print('Train:', train_images.shape, train_boxes.shape)
print('Val:', val_images.shape, val_boxes.shape)
print('Test:', test_images.shape)
display(train_boxes.head())

In [ ]:
print(train_boxes.isna().sum())
print(val_boxes.isna().sum())
print(test_images.isna().sum())

In [ ]:
dd = pd.DataFrame({
    'column': ['x', 'y', 'w', 'h', 'category_id', 'orig_category_id', 'iscrowd',
               'truncation', 'occlusion', 'track_id', 'video', 'frame_id'],
    'description': ['left edge, px', 'top edge, px', 'box width, px', 'box height, px',
                    'merged class, 1-8', 'original VisDrone class', '1 = ignore region, not an object',
                    '1 = object cut by the frame edge', '0 none, 1 partial, 2 heavy',
                    'same object across frames', 'clip name', 'frame number in the clip'],
}).set_index('column')
dd['dtype'] = train_boxes.dtypes.astype(str)
dd['missing_%'] = (train_boxes.isna().mean() * 100).round(2)
dd[['dtype', 'missing_%', 'description']]

Before any statistics, look at real frames. Green boxes are labelled objects, red boxes are `iscrowd=1` rows.

In [ ]:
clips = train_images.video.drop_duplicates().sample(6, random_state=42)
fig, axes = plt.subplots(2, 3, figsize=(16, 6.5))

for ax, clip in zip(axes.flatten(), clips):
    frame = train_images[train_images.video == clip].iloc[0]
    boxes = train_boxes[train_boxes.image_id == frame.image_id]
    image = cv2.cvtColor(cv2.imread(str(DATA / 'train' / frame.file_name)), cv2.COLOR_BGR2RGB)
    draw_boxes(image, boxes[boxes.iscrowd == 0], (0, 220, 0))
    draw_boxes(image, boxes[boxes.iscrowd == 1], (230, 40, 40))
    ax.imshow(image)
    ax.set_title(f'{clip} ({frame.width}x{frame.height})', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

### What are the `iscrowd` rows?
`track_id`, `truncation` and `occlusion` are missing on the same rows, about a third of the table. Check how those rows split by class.

In [ ]:
pd.crosstab(train_boxes.iscrowd, train_boxes.category_id)

Every class has exactly the same number of `iscrowd=1` rows. Real objects cannot do that, so these look like the same regions copied once per class. Count the unique regions.

In [ ]:
crowd = train_boxes[train_boxes.iscrowd == 1]
regions = crowd.drop_duplicates(['image_id', 'x', 'y', 'w', 'h'])

print('rows:', len(crowd), ' unique regions:', len(regions), ' copies:', len(crowd) / len(regions))
print(crowd.orig_category_id.value_counts())

Confirmed: 63,843 regions, each repeated 8 times. They come from the original VisDrone classes 0 (ignored region) and 11 (others). So non-target objects and areas the annotators skipped are both stored as `iscrowd=1`, and neither is a training target. The other original classes are merged into the 8 target classes (for example pedestrian and people become person).

Do the YOLO `.txt` labels contain them? Compare the number of lines with the real boxes on every frame.

In [ ]:
real_per_frame = train_boxes[train_boxes.iscrowd == 0].groupby('image_id').size()
expected = train_images.image_id.map(real_per_frame).fillna(0).astype(int)

lines = [len((DATA / 'train' / 'labels' / r.video / f'{r.frame_id:07d}.txt').read_text().splitlines())
         for r in tqdm(train_images.itertuples(), total=len(train_images))]

print('frames where labels differ from real boxes:', (np.array(lines) != expected).sum())
print('frames without any object:', (expected == 0).sum())

The labels already skip the ignore regions, so they can be used as they are. Next, check the real boxes for impossible values.

In [ ]:
def add_geometry(df):
    df = df[df.iscrowd == 0].copy()
    df['rel_size'] = np.sqrt(df.w * df.h / (df.width * df.height))
    df['cx'] = (df.x + df.w / 2) / df.width
    df['cy'] = (df.y + df.h / 2) / df.height
    df['class_name'] = df.category_id.map(lambda i: CLASSES[i - 1])
    return df


real_train = add_geometry(train_boxes)
real_val = add_geometry(val_boxes)

for name, df in [('train', real_train), ('val', real_val)]:
    outside = (df.x < 0) | (df.y < 0) | (df.x + df.w > df.width) | (df.y + df.h > df.height)
    print(name)
    print('  non-positive size:', ((df.w <= 0) | (df.h <= 0)).sum())
    print('  outside the frame:', outside.sum())
    print('  duplicate track in a frame:', df.duplicated(['video', 'frame_id', 'track_id']).sum())
    print('  exact duplicate boxes:', df.duplicated(['image_id', 'category_id', 'x', 'y', 'w', 'h']).sum())
    print('  smallest side, px:', np.minimum(df.w, df.h).min())

There is nothing to fix here: the real labels have no impossible values, and even the smallest boxes are valid objects.

### Box size
Drone footage has very small objects, and this decides the input size of the model. Compare sizes as a share of the frame, because frame resolutions differ.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
columns = [('w', 'box width, px'), ('h', 'box height, px'), ('rel_size', 'sqrt(box area / frame area)')]

for ax, (col, title) in zip(axes, columns):
    both = pd.concat([real_train[col], real_val[col]])
    bins = np.geomspace(both.quantile(0.001), both.quantile(0.999), 40)
    ax.hist(real_train[col], bins=bins, density=True, histtype='step', lw=2, color=COLORS['train'], label='train')
    ax.hist(real_val[col], bins=bins, density=True, histtype='step', lw=2, color=COLORS['val'], label='val')
    ax.set_xscale('log')
    ax.set_title(title)
    ax.set_yticks([])
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
order = real_train.groupby('class_name').rel_size.median().sort_values().index
sns.boxplot(data=real_train, x='rel_size', y='class_name', order=order, showfliers=False,
            color=COLORS['train'], log_scale=True, ax=ax)
ax.set_xlabel('sqrt(box area / frame area)')
ax.set_ylabel('')
ax.set_title('Box size by class', loc='left')
plt.tight_layout()
plt.show()

for imgsz in (640, 960, 1280):
    side = np.sqrt(real_train.w * real_train.h) * imgsz / np.maximum(real_train.width, real_train.height)
    print(f'imgsz {imgsz}: {(side < 8).mean():.1%} of boxes under 8 px, {(side < 16).mean():.1%} under 16 px')

Train and val have almost the same size distribution, so val is a fair check for object size. Objects are tiny: at `imgsz=640`, 54.2% of boxes are under 16 px, compared with 24.0% at 960 and 9.8% at 1280. Person, bicycle and motor are the smallest classes, and bus is the largest.

Decision: `imgsz=960` for all detectors, with 1280 tested as an experiment. Anchor sizes for the anchor-based models come from these distributions.

### Classes

In [ ]:
share = pd.DataFrame({'train': real_train.class_name.value_counts(normalize=True),
                      'val': real_val.class_name.value_counts(normalize=True)}).sort_values('train')

fig, ax = plt.subplots(figsize=(9, 4.5))
share.plot.barh(ax=ax, color=[COLORS['train'], COLORS['val']], width=0.8)
ax.set_xlabel('share of boxes')
ax.set_ylabel('')
ax.set_title('Class share in train and val', loc='left')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print(real_train.class_name.value_counts())
print('car / bus:', round((real_train.class_name == 'car').sum() / (real_train.class_name == 'bus').sum(), 1))

In [ ]:
occluded = real_train.groupby('class_name').occlusion.apply(lambda s: (s > 0).mean()).sort_values()

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(occluded.index, occluded.values, color=COLORS['train'])
for y, v in enumerate(occluded.values):
    ax.text(v, y, f' {v:.0%}', va='center', fontsize=9)
ax.set_xlim(0, 1)
ax.set_title('Share of train boxes that are partly or heavily occluded', loc='left')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

Car and person make up about 75% of train boxes, and there are 52 cars for every bus. Val has the opposite order (person 44%, car 28%), and bus has only 264 val boxes, so its val AP will be noisy. Results should be read per class, not only as one number.

Between 43% and 65% of the boxes in every class are partly or heavily occluded, with truck the highest. Scenes are crowded, so an aggressive NMS could remove real neighbouring objects.

### Where objects are and how many

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].hist2d(real_train.cx, real_train.cy, bins=60, range=[[0, 1], [0, 1]], cmap='Blues')
axes[0].invert_yaxis()
axes[0].set_title('Box centres in the frame (train)')
axes[0].set_xlabel('x / width')
axes[0].set_ylabel('y / height')

per_frame = {'train': real_train.groupby('image_id').size(), 'val': real_val.groupby('image_id').size()}
bins = np.arange(0, 301, 10)
for name, counts in per_frame.items():
    axes[1].hist(counts, bins=bins, density=True, histtype='step', lw=2, color=COLORS[name], label=name)
axes[1].set_title('Objects per frame')
axes[1].set_xlabel('objects')
axes[1].set_yticks([])
axes[1].legend()

plt.tight_layout()
plt.show()

print(per_frame['train'].describe().round(1))

Objects gather around the centre and upper middle of the frame, because the camera looks down at an angle. A horizontal flip keeps this geometry, a vertical flip does not, so vertical flips stay off.

The median train frame has 43 objects and the densest has 180. That is below the 300 queries RT-DETR uses by default and the 500 detections scored per image, so neither limit should cut real objects. Val has more sparse frames than train, with a clear peak at 10-20 objects.

### How much clips differ
A dataset average can hide large differences between sequences, so repeat the main statistics for each clip.

In [ ]:
per_clip = real_train.groupby('video').agg(boxes=('id', 'size'), frames=('image_id', 'nunique'),
                                           box_size=('rel_size', 'median'))
per_clip['objects_per_frame'] = per_clip.boxes / per_clip.frames
per_clip['person_share'] = real_train.class_name.eq('person').groupby(real_train.video).mean()

columns = [('objects_per_frame', 'Objects per frame'), ('box_size', 'Median box size'),
           ('person_share', 'Share of person boxes')]

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
for ax, (col, title) in zip(axes, columns):
    values = per_clip[col].sort_values().to_numpy()
    ax.bar(range(len(values)), values, width=0.8, color=COLORS['train'])
    ax.set_title(title, loc='left')
    ax.set_xlabel('clips, sorted')
    ax.set_xticks([])
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

per_clip[[c for c, _ in columns]].describe().loc[['min', '50%', 'max']].round(3)

Clips are very different. Objects per frame go from 5 to 143 (median 44), median box size from 0.019 to 0.061 of the frame, and the share of person boxes from 0% to 100%. This is why the splits are made by clip, and why one validation number depends so much on which clips are in val.

### Ignore regions
Objects inside a region are visible but not labelled, so a detector would be punished for finding them. The usual fix is to paint the regions out. Before doing that, check how much of the frame they cover and whether real boxes sit inside them.

In [ ]:
pairs = real_train[['id', 'image_id', 'x', 'y', 'w', 'h']].merge(
    regions[['image_id', 'x', 'y', 'w', 'h']], on='image_id', suffixes=('', '_r'))
overlap_w = (np.minimum(pairs.x + pairs.w, pairs.x_r + pairs.w_r) - np.maximum(pairs.x, pairs.x_r)).clip(lower=0)
overlap_h = (np.minimum(pairs.y + pairs.h, pairs.y_r + pairs.h_r) - np.maximum(pairs.y, pairs.y_r)).clip(lower=0)
pairs['covered'] = overlap_w * overlap_h / (pairs.w * pairs.h)
covered = pairs.groupby('id').covered.max()

coverage = (regions.w * regions.h / (regions.width * regions.height)).groupby(regions.image_id).sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(coverage.clip(upper=0.3), bins=30, color=COLORS['train'])
axes[0].set_title('Share of the frame inside ignore regions')
axes[0].set_xlabel('share of frame')
axes[0].set_ylabel('frames')
axes[1].hist(covered[covered > 0], bins=20, color=COLORS['train'])
axes[1].set_title('Real boxes touching a region: how much is covered')
axes[1].set_xlabel('share of the box inside a region')
axes[1].set_ylabel('boxes')
plt.tight_layout()
plt.show()

print('frames with regions:', coverage.size, 'of', len(train_images))
print('mean frame coverage:', round(coverage.mean(), 4))
print('real boxes touching a region:', (covered > 0).sum())
print('real boxes more than half covered:', (covered > 0.5).sum())

Regions appear on 18,090 of 24,201 frames (75%), but they cover only about 3.2% of a frame on average, so painting them removes little. However, 38,801 real boxes touch a region and 11,733 are more than half inside one. The peak at 1.0 shows many labelled objects sit fully inside a region.

Decision: paint the regions, then copy every labelled box back.

### Train vs test
Test has no labels, so compare what is available: resolution, clip length and the images themselves.

In [ ]:
def resolution_share(df):
    return (df.width.astype(str) + 'x' + df.height.astype(str)).value_counts(normalize=True)


resolutions = pd.DataFrame({'train': resolution_share(train_images), 'val': resolution_share(val_images),
                            'test': resolution_share(test_images)}).fillna(0)
resolutions = resolutions.sort_values('test', ascending=False)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(resolutions, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, cbar=False, ax=ax)
ax.set_title('Share of frames at each resolution', loc='left')
plt.tight_layout()
plt.show()

In [ ]:
lengths = {'train': train_images.groupby('video').size(), 'val': val_images.groupby('video').size(),
           'test': test_images.groupby('clip').size()}


def image_stats(files, n=300):
    rows = []
    for f in pd.Series(files).sample(n, random_state=42):
        gray = cv2.imread(str(DATA / f), cv2.IMREAD_GRAYSCALE)
        rows.append({'brightness': gray.mean(), 'contrast': gray.std()})
    return pd.DataFrame(rows)


stats = {'train': image_stats('train/' + train_images.file_name),
         'val': image_stats('val/' + val_images.file_name),
         'test': image_stats(test_images.file_name)}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
bins = np.geomspace(30, 1500, 18)
for name, values in lengths.items():
    axes[0].hist(values, bins=bins, histtype='step', lw=2, color=COLORS[name], label=name)
axes[0].set_xscale('log')
axes[0].set_title('Frames per clip')
axes[0].set_ylabel('clips')
axes[0].legend()

for ax, col in zip(axes[1:], ['brightness', 'contrast']):
    both = pd.concat([s[col] for s in stats.values()])
    bins = np.linspace(both.min(), both.max(), 25)
    for name, s in stats.items():
        ax.hist(s[col], bins=bins, density=True, histtype='step', lw=2, color=COLORS[name], label=name)
    ax.set_title(col)
    ax.set_yticks([])
    ax.legend()

plt.tight_layout()
plt.show()

print(pd.DataFrame({k: v.describe() for k, v in lengths.items()}).loc[['count', 'min', '50%', 'max']])
print(pd.DataFrame({k: v.mean() for k, v in stats.items()}).round(1))

- **Resolution:** only 1344x756 and 1920x1080 appear in both train and test. Test frames are never larger than 1920x1080, and about 16% are low resolution (680x382 or 960x540), while train goes up to 2720x1530. Frames are not resized on disk, because the model input size handles this.
- **Val:** 34% of val frames are 3840x2160, so one kind of footage has a large effect on val scores.
- **Clip length:** test clips are short (35-148 frames) compared with train (58-1,424).
- **Images:** on 300 sampled frames per split, test is darker (mean brightness 101.6 vs 120.5) and has lower contrast (42.5 vs 47.5). Brightness augmentation should help. Val forms separate groups because it has only 7 clips.

### How similar are neighbouring frames?
Video frames are not independent, so check how fast a clip changes: the mean pixel difference between a frame and the frame `gap` steps later.

In [ ]:
def small_gray(file_name):
    gray = cv2.imread(str(DATA / 'train' / file_name), cv2.IMREAD_GRAYSCALE)
    return cv2.resize(gray, (320, 180)).astype(float)


gaps = [1, 2, 5, 10, 30, 60]
long_clips = train_images.groupby('video').size().loc[lambda s: s >= 200].index
rows = []

for clip in pd.Series(long_clips).sample(8, random_state=42):
    files = train_images[train_images.video == clip].sort_values('frame_id').file_name.tolist()
    for start in np.linspace(0, len(files) - max(gaps) - 1, 4).astype(int):
        first = small_gray(files[start])
        for gap in gaps:
            rows.append({'clip': clip, 'gap': gap, 'diff': np.abs(first - small_gray(files[start + gap])).mean()})

change = pd.DataFrame(rows).groupby('gap')['diff'].median()

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(change.index, change.values, marker='o', lw=2, color=COLORS['train'])
ax.set_xscale('log')
ax.set_xticks(gaps, labels=gaps)
ax.set_ylim(0)
ax.set_xlabel('gap, frames')
ax.set_ylabel('mean pixel difference (0-255)')
ax.set_title('Frames change slowly inside a clip', loc='left')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print(change.round(1))

The difference is 6.6 at a gap of 1, 15.8 at a gap of 5 and 29.5 at a gap of 60 (8 clips, 4 start points each). A gap of 5 already reaches over half of the change seen at 60, so every 5th frame keeps most of the variety for a fifth of the training cost. This is why `TRAIN_STRIDE = 5`.

The stride does not balance the clips: a long clip still gives more frames than a short one.

## Data cleaning: `clean()`
One deterministic function for train and val. It works row by row and learns nothing from the data, so there is no leakage between folds.
- `iscrowd=1`: removed from the targets, one copy of each region kept as `ignore`;
- class ids: shifted from 1-8 to the 0-7 used in the YOLO labels.

In [ ]:
def clean(boxes: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    real = boxes[boxes.iscrowd == 0].copy()
    real['cls'] = real.category_id - 1
    ignore = boxes[boxes.iscrowd == 1].drop_duplicates(['image_id', 'x', 'y', 'w', 'h'])
    return real, ignore


train_clean, train_ignore = clean(train_boxes)
val_clean, val_ignore = clean(val_boxes)

print('Train:', len(train_clean), 'boxes,', len(train_ignore), 'ignore regions')
print('Val:', len(val_clean), 'boxes,', len(val_ignore), 'ignore regions')

## Splits by clip
A random split would put almost identical frames on both sides, so `GroupKFold` keeps each clip in one fold. The provided train and val share no clips.

In [ ]:
assert not set(train_images.video) & set(val_images.video)

gkf = GroupKFold(n_splits=5, shuffle=True, random_state=42)
train_images['fold'] = -1
for k, (_, rows) in enumerate(gkf.split(train_images, groups=train_images.video)):
    train_images.loc[train_images.index[rows], 'fold'] = k

display(train_images.groupby('fold').agg(clips=('video', 'nunique'), frames=('image_id', 'size')))

folds = real_train.assign(fold=real_train.image_id.map(train_images.set_index('image_id').fold))
fold_share = pd.crosstab(folds.fold, folds.class_name, normalize='index')[CLASSES]

fig, ax = plt.subplots(figsize=(9, 3.4))
sns.heatmap(fold_share, annot=True, fmt='.2f', cmap='Blues', cbar=False, ax=ax)
ax.set_title('Class share in each fold', loc='left')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

Folds differ a lot: car share goes from 33% to 55%, and fold size from 4,002 to 6,057 frames. A single split depends heavily on which clips it contains.

Five folds per detector do not fit the GPU budget, so all detectors are compared on the provided val split, which shares no clips with train.

In [ ]:
TRAIN_STRIDE = 5

train_subset = train_images[train_images.groupby('video').cumcount() % TRAIN_STRIDE == 0]
print('Train frames:', len(train_images), '->', len(train_subset))

subset_boxes = real_train[real_train.image_id.isin(train_subset.image_id)]
compare = pd.DataFrame({'all frames': real_train.class_name.value_counts(normalize=True),
                        'every 5th frame': subset_boxes.class_name.value_counts(normalize=True)}).sort_values('all frames')
compare['diff'] = compare['all frames'] - compare['every 5th frame']
print(compare.round(4))

Class shares change by at most 0.04 percentage points, so the subset keeps the class balance.

## Build the training folder
Ignore regions are painted grey, then every labelled box is copied back so no real object is lost. Val is painted too, otherwise detections of unlabelled objects would count as false positives. The competition does not penalise detections inside ignore regions, but test has no ignore files and stays as it is, so val scores are only a guide to the leaderboard.

Frames without regions are symlinks to the originals. The folder, together with the test frames, is saved to Drive as `prep.tar`, so a GPU runtime does not need the full dataset.

In [ ]:
PREP = Path('/content/prep' if IN_COLAB else 'prep').resolve()
MASK_VALUE = 114


def mask_frame(image, regions, objects):
    masked = image.copy()
    for x, y, w, h in regions:
        masked[y:y + h, x:x + w] = MASK_VALUE
    for x, y, w, h in objects:
        masked[y:y + h, x:x + w] = image[y:y + h, x:x + w]
    return masked


def boxes_by_image(boxes):
    return {k: v.to_numpy(int) for k, v in boxes.groupby('image_id')[['x', 'y', 'w', 'h']]}


def link_or_copy(source, target):
    # symlinks need extra rights on Windows, so fall back to a hard link or a copy
    try:
        target.symlink_to(source)
    except OSError:
        try:
            os.link(source, target)
        except OSError:
            shutil.copy(source, target)


def build_split(split, images, real, ignore):
    shutil.rmtree(PREP / split, ignore_errors=True)
    regions, objects = boxes_by_image(ignore), boxes_by_image(real)
    no_boxes = np.empty((0, 4), int)

    for r in tqdm(images.itertuples(), total=len(images), desc=split):
        label = Path(r.file_name).relative_to('images').with_suffix('.txt')
        image_path = PREP / split / r.file_name
        label_path = PREP / split / 'labels' / label
        image_path.parent.mkdir(parents=True, exist_ok=True)
        label_path.parent.mkdir(parents=True, exist_ok=True)

        source = DATA / split / r.file_name
        if r.image_id in regions:
            image = cv2.imread(str(source))
            masked = mask_frame(image, regions[r.image_id], objects.get(r.image_id, no_boxes))
            cv2.imwrite(str(image_path), masked, [cv2.IMWRITE_JPEG_QUALITY, 95])
        else:
            link_or_copy(source, image_path)
        shutil.copy(DATA / split / 'labels' / label, label_path)


build_split('train', train_subset, train_clean, train_ignore)
build_split('val', val_images, val_clean, val_ignore)

data_yaml = PREP / 'data.yaml'
data_yaml.write_text(f"path: {PREP.resolve()}\ntrain: train/images\nval: val/images\n"
                     f"nc: {len(CLASSES)}\nnames: {json.dumps(CLASSES)}\n")
print(data_yaml.read_text())

In [ ]:
if IN_COLAB:
    local_archive = Path('/content/prep.tar')
    !tar -chf {local_archive} -C {DATA} test test_images.csv -C {PREP} train val data.yaml
    size_gb = local_archive.stat().st_size / 1e9
    free_gb = shutil.disk_usage(DRIVE).free / 1e9
    print(f'Archive: {size_gb:.1f} GB, free on Drive: {free_gb:.1f} GB')
    assert size_gb < free_gb, 'Not enough space on Drive for the archive'
    shutil.copy(local_archive, f'{PREP_ARCHIVE}.part')
    os.replace(f'{PREP_ARCHIVE}.part', PREP_ARCHIVE)
    local_archive.unlink()

Check that nothing was lost: every frame has an image and a label file, and the number of boxes matches `clean()`.

In [ ]:
for split, images, real in [('train', train_subset, train_clean), ('val', val_images, val_clean)]:
    n_images = len(list((PREP / split / 'images').rglob('*.jpg')))
    labels = list((PREP / split / 'labels').rglob('*.txt'))
    n_boxes = sum(len(p.read_text().splitlines()) for p in labels)
    expected = real.image_id.isin(images.image_id).sum()
    print(f'{split}: {n_images} images, {len(labels)} labels, {n_boxes} boxes (expected {expected})')
    assert n_images == len(labels) == len(images) and n_boxes == expected

In [ ]:
area = train_ignore[train_ignore.image_id.isin(train_subset.image_id)]
area = (area.w * area.h / (area.width * area.height)).groupby(area.image_id).sum()
frame = train_subset[train_subset.image_id == area.idxmax()].iloc[0]
boxes = train_clean[train_clean.image_id == frame.image_id]
regions_in_frame = train_ignore[train_ignore.image_id == frame.image_id]

before = cv2.cvtColor(cv2.imread(str(DATA / 'train' / frame.file_name)), cv2.COLOR_BGR2RGB)
after = cv2.cvtColor(cv2.imread(str(PREP / 'train' / frame.file_name)), cv2.COLOR_BGR2RGB)
draw_boxes(before, regions_in_frame, (230, 40, 40))
draw_boxes(after, boxes, (0, 220, 0))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].imshow(before)
axes[0].set_title('Before: ignore regions in red')
axes[1].imshow(after)
axes[1].set_title('After: regions painted grey, labelled boxes in green')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Resizing, normalization and augmentation
- **Resizing:** frames stay at their original size on disk. Ultralytics letterboxes them to `imgsz` keeping the aspect ratio; torchvision models resize them to the same 960 px long side.
- **Normalization:** pixel values are divided by 255 inside the framework.
- **Augmentation:** Ultralytics defaults: mosaic (turned off for the last 10 epochs), horizontal flip with probability 0.5, scale 0.5, translate 0.1, and HSV jitter (`hsv_v=0.4` covers the brightness gap with test). Torchvision models use horizontal flips. `flipud` stays 0, as explained above.
- **Reproducibility:** `seed=42` for training, `random_state=42` for sampling; `TRAIN_STRIDE`, `TRAIN_STEP`, `IMGSZ` and `EPOCHS` are set in the notebook.

## Preprocessing summary

| Check | Result |
|---|---|
| Missing values | Only on the `iscrowd=1` rows |
| Impossible values: box size, frame bounds, duplicates | None in real boxes |
| `iscrowd=1` rows | Ignore regions copied 8 times, removed from targets and painted grey |
| `clean()` | Same function for train and val |
| Imputation, one-hot encoding | Not needed: boxes have no missing values and classes are already numbers |
| Train vs test | Different resolutions; test clips are shorter and darker |
| Splits | `GroupKFold` by clip, no clip on both sides |
| Subsampling | `TRAIN_STRIDE = 5`: 24,201 -> 4,868 frames |

## Detector setup
All detectors are trained under one protocol and scored by one evaluator, so differences come from the model and not from the setup.

In [ ]:
PREP = Path('/content/prep' if IN_COLAB else 'prep').resolve()
VAL_JSON = Path(DRIVE) / 'val_annotations_coco.json'

if not (PREP / 'test_images.csv').exists() and Path(PREP_ARCHIVE).exists():
    PREP.mkdir(exist_ok=True)
    !tar -xf {PREP_ARCHIVE} -C {PREP}

DATA = Path(path) if 'path' in globals() else PREP
if not VAL_JSON.exists():
    source = DATA / 'val' / 'annotations_coco.json'
    if not source.exists():
        os.environ['KAGGLE_API_TOKEN'] = os.environ.get('KAGGLE_API_TOKEN') or getpass('Kaggle API token: ')
        source = kagglehub.competition_download(COMPETITION, path='val/annotations_coco.json')
    shutil.copy(source, VAL_JSON)

test_images = pd.read_csv(DATA / 'test_images.csv')
print('Device:', 'GPU' if torch.cuda.is_available() else 'CPU', ' training folder:', (PREP / 'data.yaml').exists())

In [ ]:
IMGSZ = 960
MIN_SIDE = IMGSZ * 9 // 16
TRAIN_STEP = 3
EPOCHS = 1 if SMOKE_TEST else 4
BATCH = 4
LR = 0.005
MAX_DET = 500
SCORE_THRESHOLD = 0.01
NMS_IOU = 0.7
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
WORKERS = min(8, os.cpu_count()) if DEVICE == 'cuda' and os.name != 'nt' else 0


def label_path(image_path):
    return Path(str(image_path).replace(f'{os.sep}images{os.sep}', f'{os.sep}labels{os.sep}')).with_suffix('.txt')


def read_labels(image_path):
    path = label_path(image_path)
    return np.loadtxt(path, ndmin=2).reshape(-1, 5) if path.stat().st_size else np.zeros((0, 5))


train_frames = sorted((PREP / 'train' / 'images').rglob('*.jpg'))[::TRAIN_STEP]
val_coco = json.loads(VAL_JSON.read_text())
val_frames = [(im['id'], PREP / 'val' / im['file_name']) for im in val_coco['images']]
test_frames = [(i, DATA / f) for i, f in enumerate(test_images.file_name)]
if SMOKE_TEST:
    train_frames = train_frames[::len(train_frames) // 24][:24]
    val_frames = val_frames[::len(val_frames) // 12][:12]

(PREP / 'train_protocol.txt').write_text(''.join(f'{p}\n' for p in train_frames))
(PREP / 'val_protocol.txt').write_text(''.join(f'{p}\n' for _, p in val_frames))
protocol_yaml = PREP / 'protocol.yaml'
protocol_yaml.write_text(f'path: {PREP}\ntrain: train_protocol.txt\nval: val_protocol.txt\n'
                         f'nc: {len(CLASSES)}\nnames: {json.dumps(CLASSES)}\n')
print(f'{len(train_frames)} train frames x {EPOCHS} epochs, {len(val_frames)} val frames')

### Shared evaluator
COCO mAP, AP50, AP75, AP by object size and AP per class on the val frames. `AP_small` covers boxes under 32x32 px in the original frame. Ignore regions stay in the ground truth as `iscrowd=1`, so detections inside them are not penalised, as on the leaderboard, and `maxDets` is 500, the leaderboard limit.

Two additions, because val differs from test:
- `mAP_testlike`: mAP on the 4 val clips that are at most 1920 px wide (1,286 of 2,846 frames). The other 3 clips are 2688-3840 px wide, and no test frame is wider than 1920 px. This is `SELECT_METRIC`, which picks the final model.
- `clip_<name>`: mAP of each of the 7 val clips. `paired` compares two runs clip by clip, which shows whether a gain holds on most clips or comes from one of them.

Every result goes to `experiment_log.csv` on Drive with the val predictions next to the weights, so finished runs are not repeated.

In [ ]:
ground_truth = COCO(str(VAL_JSON))
LOG = Path(RUNS) / 'experiment_log.csv'
VAL_INFO = {im['id']: im for im in val_coco['images']}
VAL_IDS = [image_id for image_id, _ in val_frames]
TEST_LIKE_IDS = [i for i in VAL_IDS if VAL_INFO[i]['width'] <= 1920]
CLIP_IDS = {}
for image_id in VAL_IDS:
    CLIP_IDS.setdefault(VAL_INFO[image_id]['video'], []).append(image_id)


def coco_eval(results, image_ids):
    with contextlib.redirect_stdout(io.StringIO()):
        evaluation = COCOeval(ground_truth, results, 'bbox')
        evaluation.params.maxDets = [1, 100, MAX_DET]
        evaluation.params.imgIds = image_ids
        evaluation.evaluate()
        evaluation.accumulate()
        evaluation.summarize()
    return evaluation


def coco_metrics(rows):
    results = ground_truth.loadRes(rows)
    evaluation = coco_eval(results, VAL_IDS)
    names = ['mAP', 'AP50', 'AP75', 'AP_small', 'AP_medium', 'AP_large']
    metrics = dict(zip(names, [float(x) for x in evaluation.stats[:6]]))
    precision = evaluation.eval['precision'][:, :, :, 0, -1]
    for k, name in enumerate(CLASSES):
        values = precision[:, :, k]
        metrics[f'AP_{name}'] = float(values[values > -1].mean()) if (values > -1).any() else 0.0
    if TEST_LIKE_IDS:
        stats = coco_eval(results, TEST_LIKE_IDS).stats
        metrics.update(mAP_testlike=float(stats[0]), AP50_testlike=float(stats[1]), AP_small_testlike=float(stats[3]))
    for clip, ids in CLIP_IDS.items():
        metrics[f'clip_{clip}'] = float(coco_eval(results, ids).stats[0])
    return metrics


def load_log():
    return pd.read_csv(LOG) if LOG.exists() else pd.DataFrame(columns=['run', 'section'])


def show(name):
    r = load_log().set_index('run').loc[name]
    like = f'  test-like {r.mAP_testlike:.3f}' if pd.notna(r.get('mAP_testlike')) else ''
    print(f'{name}: mAP {r.mAP:.3f}{like}  AP50 {r.AP50:.3f}  AP_small {r.AP_small:.3f}  {r.ms_per_frame:.1f} ms/frame')


def finish(name, section, settings, rows, ms, minutes, **extra):
    assert drive_ok(), 'Google Drive is not mounted'
    (Path(RUNS) / name).mkdir(parents=True, exist_ok=True)
    np.save(Path(RUNS) / name / 'val_predictions.npy', rows)
    row = {'run': name, 'section': section, 'settings': settings, **coco_metrics(rows),
           'ms_per_frame': ms, 'train_minutes': minutes, **extra}
    log = load_log()
    pd.concat([log[log.run != name], pd.DataFrame([row])], ignore_index=True).to_csv(LOG, index=False)
    show(name)


def should_run(name):
    if ONLY and name not in ONLY:
        return False
    if FORCE or name not in set(load_log().run):
        return True
    show(name)
    return False


def best_run(section):
    log = load_log()
    rows = log[(log.section == section) & ~log.run.str.endswith('-base')]
    return rows.nlargest(1, 'mAP').run.iloc[0] if len(rows) else None


def paired(runs, reference):
    log = load_log().set_index('run')
    clips = log.filter(like='clip_')
    gain = clips.loc[list(runs)] - clips.loc[reference]
    return pd.DataFrame({SELECT_METRIC: log.loc[list(runs), SELECT_METRIC], 'mean clip gain': gain.mean(axis=1),
                         'clips better': (gain > 0).sum(axis=1), 'clips': gain.notna().sum(axis=1)}).round(3)


def unfinished(last):
    return torch.load(last, map_location='cpu', weights_only=False).get('optimizer') is not None

Runs logged before these metrics existed get them from their saved val predictions. This needs no GPU.

In [ ]:
def backfill_metrics():
    log = load_log()
    if 'mAP' not in log:
        return
    if 'mAP_testlike' not in log:
        log['mAP_testlike'] = np.nan
    missing = log.run[log.mAP.notna() & log.mAP_testlike.isna()].tolist()
    for run in tqdm(missing, desc='backfill', leave=False):
        file = Path(RUNS) / run / 'val_predictions.npy'
        if file.exists():
            for key, value in coco_metrics(np.load(file)).items():
                log.loc[log.run == run, key] = value
    if missing:
        assert drive_ok(), 'Google Drive is not mounted'
        log.to_csv(LOG, index=False)


backfill_metrics()
display(load_log().set_index('run').filter(regex='^(mAP|mAP_testlike|clip_.*)$').round(3))

### Training runners
An interrupted Ultralytics run continues from `last.pt`, and a torchvision run continues from `checkpoint.pt`, saved after every epoch. Torchvision keeps up to 500 detections per frame instead of the default 100 for Faster R-CNN and 300 for RetinaNet, because the densest train frame has 180 objects.

In [ ]:
MODEL_KINDS = {'ultralytics', 'torchvision'}
TORCHVISION_BUILDERS = {}
SIZES = {'val': {im['id']: (im['width'], im['height']) for im in val_coco['images']},
         'test': dict(enumerate(zip(test_images.width, test_images.height)))}
ORDER = {'val': {im['id']: (im['video'], im['frame_id']) for im in val_coco['images']},
         'test': dict(enumerate(zip(test_images.clip, test_images.frame)))}


def to_rows(image_id, boxes, scores, classes):
    # one row per box: image id, x, y, w, h, score, 1-based class
    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4).copy()
    boxes[:, 2:] -= boxes[:, :2]
    return np.column_stack([np.full(len(boxes), image_id), boxes, scores, np.asarray(classes) + 1])


def by_image(rows):
    rows = rows[np.argsort(rows[:, 0], kind='stable')]
    ids, starts = np.unique(rows[:, 0].astype(int), return_index=True)
    return dict(zip(ids.tolist(), np.split(rows, starts[1:])))


def tile_windows(width, height, grid, overlap=0.2):
    tile_w = math.ceil(width / (grid - (grid - 1) * overlap))
    tile_h = math.ceil(height / (grid - (grid - 1) * overlap))
    xs = np.linspace(0, width - tile_w, grid).round().astype(int)
    ys = np.linspace(0, height - tile_h, grid).round().astype(int)
    return [(x, y, x + tile_w, y + tile_h) for y in ys for x in xs]


def inference_name(spec):
    name = f"{spec['weights']}-{spec['checkpoint']}-{spec['imgsz']}-conf{spec['conf']:g}-nms{spec['iou']:g}"
    return name + ('-tiles' if spec['tiles'] else '') + ('-tta' if spec['augment'] else '')


def ultralytics_spec(weights, model='yolo', checkpoint='last', imgsz=IMGSZ, conf=SCORE_THRESHOLD, iou=NMS_IOU,
                     augment=False, tiles=0, run=None):
    spec = dict(kind='ultralytics', weights=weights, model=model, checkpoint=checkpoint, imgsz=imgsz,
                conf=conf, iou=iou, augment=augment, tiles=tiles)
    spec['run'] = run or inference_name(spec)
    return spec


def ultralytics_predict(spec, frames):
    Model = RTDETR if spec['model'] == 'rtdetr' else YOLO
    model = Model(str(Path(RUNS) / spec['weights'] / 'weights' / f"{spec['checkpoint']}.pt"))
    args = dict(imgsz=spec['imgsz'], conf=spec['conf'], iou=spec['iou'], max_det=MAX_DET,
                augment=spec['augment'], verbose=False)
    parts, seconds = [], 0.0
    for image_id, path in tqdm(frames, leave=False):
        image = cv2.imread(str(path))
        height, width = image.shape[:2]
        windows = [(0, 0, width, height)]
        if spec['tiles'] and max(width, height) > spec['imgsz']:
            windows += tile_windows(width, height, spec['tiles'])
        results = model.predict([image[y0:y1, x0:x1] for x0, y0, x1, y1 in windows], **args)
        boxes, scores, classes = [], [], []
        for (x0, y0, _, _), result in zip(windows, results):
            seconds += result.speed['inference'] / 1000
            boxes.append(result.boxes.xyxy.float().cpu() + torch.tensor([x0, y0, x0, y0], dtype=torch.float32))
            scores.append(result.boxes.conf.float().cpu())
            classes.append(result.boxes.cls.long().cpu())
        boxes, scores, classes = torch.cat(boxes), torch.cat(scores), torch.cat(classes)
        if len(windows) > 1:
            keep = batched_nms(boxes, scores, classes, spec['iou'])[:MAX_DET]
            boxes, scores, classes = boxes[keep], scores[keep], classes[keep]
        parts.append(to_rows(image_id, boxes.numpy(), scores.numpy(), classes.numpy()))
    return np.concatenate(parts), 1000 * seconds / len(frames)


def predict_rows(spec, split):
    # predictions of any spec (one model, an ensemble or a temporal step) on 'val' or 'test'
    frames = val_frames if split == 'val' else test_frames
    if spec['kind'] == 'ultralytics':
        return ultralytics_predict(spec, frames)
    if spec['kind'] == 'torchvision':
        model = TORCHVISION_BUILDERS[spec['weights']]()
        model.load_state_dict(torch.load(Path(RUNS) / spec['weights'] / 'model.pt', map_location='cpu'))
        rows, ms = torchvision_rows(model, frames)
        del model
        torch.cuda.empty_cache()
        return rows, ms
    children = spec['members'] if spec['kind'] == 'ensemble' else [spec['base']]
    rows = [cached_rows(child, split) for child in children]
    log = load_log().set_index('run')
    ms = sum(log.ms_per_frame.get(child['run'], np.nan) for child in children)
    if spec['kind'] == 'ensemble':
        return wbf_rows(rows, split, spec['weights'], spec['iou_thr']), ms
    return temporal_rows(rows[0], split, spec['window'], spec['weight'], spec['iou_thr']), ms


def cached_rows(spec, split):
    file = Path(RUNS) / spec['run'] / f'{split}_predictions.npy'
    if file.exists():
        return np.load(file)
    rows, _ = predict_rows(spec, split)
    file.parent.mkdir(parents=True, exist_ok=True)
    np.save(file, rows)
    return rows


def spec_of(run):
    row = load_log().set_index('run').loc[run]
    if isinstance(row.get('spec'), str):
        return json.loads(row['spec'])
    if row.section in ('two-stage', 'one-stage anchor'):
        return dict(kind='torchvision', run=run, weights=run)
    checkpoint = row.checkpoint if isinstance(row.get('checkpoint'), str) else 'last'
    return ultralytics_spec(row.weights, 'rtdetr' if row.section == 'query-based' else 'yolo', checkpoint,
                            int(row.imgsz), augment=str(row.augment) == 'True', run=run)


def run_spec(spec, settings, section='pipeline'):
    if not should_run(spec['run']):
        return
    if spec['kind'] in MODEL_KINDS and DEVICE != 'cuda' and not SMOKE_TEST:
        print('Needs a GPU runtime:', spec['run'])
        return
    rows, ms = predict_rows(spec, 'val')
    finish(spec['run'], section, settings, rows, ms, np.nan, spec=json.dumps(spec),
           **{k: spec[k] for k in ('weights', 'imgsz', 'augment', 'checkpoint') if k in spec})


# Colab trains on the local disk and copies to Drive after every epoch; locally runs are written to RUNS directly
LOCAL_RUNS = Path('/content/runs') if IN_COLAB else Path(RUNS)


def print_epoch(trainer):
    if trainer.epoch >= trainer.epochs:
        return
    minutes = (time.time() - trainer.train_time_start) / 60
    print(f"epoch {trainer.epoch + 1}/{trainer.epochs}: val mAP {trainer.metrics.get('metrics/mAP50-95(B)', 0):.3f}, "
          f"{minutes:.1f} min")


def with_callbacks(model, weights):
    target = Path(RUNS) / weights

    def sync(trainer):
        assert drive_ok(), 'Google Drive is not mounted'
        if Path(trainer.save_dir).resolve() != target.resolve():
            shutil.copytree(trainer.save_dir, target, dirs_exist_ok=True)

    model.add_callback('on_fit_epoch_end', print_epoch)
    model.add_callback('on_model_save', sync)
    model.add_callback('on_train_end', sync)
    return model


def run_ultralytics(name, section, settings, make_model=None, weights=None, data=None, imgsz=IMGSZ,
                    epochs=None, batch=8, augment=False, checkpoint='last', evaluate=True, **train_args):
    if not should_run(name):
        return
    weights = weights or name
    Model = RTDETR if section == 'query-based' else YOLO
    last = Path(RUNS) / weights / 'weights' / 'last.pt'
    if make_model is not None and TRAIN:
        data, epochs = (protocol_yaml, 1) if SMOKE_TEST else (data or protocol_yaml, epochs or EPOCHS)
        if not last.exists():
            with_callbacks(make_model(), weights).train(
                data=str(data), imgsz=imgsz, epochs=epochs, batch=batch, seed=SEED, plots=False, project=str(LOCAL_RUNS),
                name=weights, exist_ok=True, **{'warmup_epochs': 1.0, 'workers': 2 if IN_COLAB else min(8, os.cpu_count()), **train_args})
        elif unfinished(last):
            local = LOCAL_RUNS / weights
            if not (local / 'weights' / 'last.pt').exists():
                shutil.copytree(last.parent.parent, local, dirs_exist_ok=True)
            with_callbacks(Model(str(local / 'weights' / 'last.pt')), weights).train(resume=True)
    if not last.exists() or unfinished(last):
        print('Not trained yet:', name)
        return
    if not evaluate:
        return
    log = pd.read_csv(Path(RUNS) / weights / 'results.csv')
    minutes = log['time'].iloc[-1] / 60 if 'time' in log else float('nan')
    spec = ultralytics_spec(weights, 'rtdetr' if section == 'query-based' else 'yolo', checkpoint, imgsz,
                            augment=augment, run=name)
    rows, ms = predict_rows(spec, 'val')
    finish(name, section, settings, rows, ms, minutes, spec=json.dumps(spec), weights=weights, imgsz=imgsz,
           augment=augment, checkpoint=checkpoint)

In [ ]:
class FrameDataset(Dataset):
    def __init__(self, images):
        self.images = images

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        path = self.images[index]
        image = read_image(str(path)).float() / 255
        height, width = image.shape[1:]
        labels = read_labels(path)
        cx, cy, w, h = (labels[:, 1:] * [width, height, width, height]).T
        boxes = torch.tensor(np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], axis=1), dtype=torch.float32)
        if torch.rand(1) < 0.5:
            image = image.flip(2)
            boxes[:, [0, 2]] = width - boxes[:, [2, 0]]
        return image, {'boxes': boxes.reshape(-1, 4), 'labels': torch.tensor(labels[:, 0] + 1, dtype=torch.int64)}


def collate(batch):
    return tuple(zip(*batch))

In [ ]:
def giou_fastrcnn_loss(class_logits, box_regression, labels, regression_targets):
    labels = torch.cat(labels)
    targets = torch.cat(regression_targets)
    classification = nn.functional.cross_entropy(class_logits, labels)
    positive = torch.where(labels > 0)[0]
    deltas = box_regression.reshape(len(labels), -1, 4)[positive, labels[positive]]

    def to_box(delta):
        dx, dy, dw, dh = (delta / delta.new_tensor([10, 10, 5, 5])).unbind(1)
        w, h = dw.clamp(max=math.log(1000 / 16)).exp(), dh.clamp(max=math.log(1000 / 16)).exp()
        return torch.stack([dx - w / 2, dy - h / 2, dx + w / 2, dy + h / 2], dim=1)

    box = generalized_box_iou_loss(to_box(deltas), to_box(targets[positive]), reduction='sum')
    return classification, box / labels.numel()


original_fastrcnn_loss = roi_heads.fastrcnn_loss
original_focal_loss = torchvision.ops.sigmoid_focal_loss


def build_faster_rcnn(anchors=(32, 64, 128, 256, 512), proposals=1000, box_loss='smooth_l1', min_size=MIN_SIDE,
                      max_size=IMGSZ):
    roi_heads.fastrcnn_loss = giou_fastrcnn_loss if box_loss == 'giou' else original_fastrcnn_loss
    model = fasterrcnn_resnet50_fpn_v2(weights='DEFAULT', min_size=min_size, max_size=max_size,
                                       rpn_post_nms_top_n_test=proposals,
                                       box_detections_per_img=MAX_DET, box_score_thresh=SCORE_THRESHOLD)
    model.rpn.anchor_generator = AnchorGenerator(tuple((size,) for size in anchors),
                                                 ((0.5, 1.0, 2.0),) * len(anchors))
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 9)
    return model


def build_retinanet(anchors=(32, 64, 128, 256, 512), alpha=0.25, gamma=2.0):
    retinanet.sigmoid_focal_loss = partial(original_focal_loss, alpha=alpha, gamma=gamma)
    sizes = tuple((x, int(x * 2 ** (1 / 3)), int(x * 2 ** (2 / 3))) for x in anchors)
    generator = AnchorGenerator(sizes, ((0.5, 1.0, 2.0),) * len(sizes))
    model = retinanet_resnet50_fpn_v2(weights='DEFAULT', min_size=MIN_SIDE, max_size=IMGSZ,
                                      detections_per_img=MAX_DET, score_thresh=SCORE_THRESHOLD, topk_candidates=2000)
    model.anchor_generator = generator
    model.head.classification_head = RetinaNetClassificationHead(
        model.backbone.out_channels, generator.num_anchors_per_location()[0], 9,
        norm_layer=partial(nn.GroupNorm, 32))
    return model

In [ ]:
def train_torchvision(model, folder, epochs=None, lr=LR, frames=None, batch=BATCH):
    epochs = 1 if SMOKE_TEST else (epochs or EPOCHS)
    frames = train_frames if SMOKE_TEST or frames is None else frames
    checkpoint = folder / 'checkpoint.pt'
    loader = DataLoader(FrameDataset(frames), batch_size=batch, shuffle=True, collate_fn=collate,
                        num_workers=WORKERS)
    model.to(DEVICE).train()
    optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                                lr=lr, momentum=0.9, weight_decay=5e-4)
    steps = epochs * len(loader)
    warmup = min(500, max(1, steps // 10))
    scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [
        torch.optim.lr_scheduler.LinearLR(optimizer, 0.01, total_iters=warmup),
        torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, max(1, steps - warmup))], milestones=[warmup])
    scaler = torch.amp.GradScaler(enabled=DEVICE == 'cuda')
    state = {'epoch': 0, 'minutes': 0.0, 'loss': []}

    if checkpoint.exists():
        saved = torch.load(checkpoint, map_location='cpu', weights_only=False)
        model.load_state_dict(saved['model'])
        optimizer.load_state_dict(saved['optimizer'])
        scheduler.load_state_dict(saved['scheduler'])
        scaler.load_state_dict(saved['scaler'])
        state = saved['state']

    for epoch in range(state['epoch'], epochs):
        torch.manual_seed(SEED + epoch)
        start, losses = time.time(), []
        for images, targets in tqdm(loader, desc=f'epoch {epoch + 1}/{epochs}', leave=False):
            images = [image.to(DEVICE) for image in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            with torch.autocast(DEVICE, enabled=DEVICE == 'cuda'):
                loss = sum(model(images, targets).values())
            optimizer.zero_grad()
            if torch.isfinite(loss):
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                losses.append(loss.item())
            scheduler.step()
        seconds = time.time() - start
        state = {'epoch': epoch + 1, 'minutes': state['minutes'] + seconds / 60,
                 'loss': state['loss'] + [float(np.mean(losses))]}
        print(f'epoch {epoch + 1}: loss {state["loss"][-1]:.3f}, {len(frames) / seconds:.1f} frames/s')
        torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                    'scaler': scaler.state_dict(), 'state': state}, f'{checkpoint}.part')
        os.replace(f'{checkpoint}.part', checkpoint)
    return state


@torch.no_grad()
def torchvision_rows(model, frames=None):
    frames = val_frames if frames is None else frames
    model.to(DEVICE).eval()
    parts, seconds = [], 0.0
    for start in tqdm(range(0, len(frames), BATCH), leave=False):
        batch = frames[start:start + BATCH]
        images = [(read_image(str(path)).float() / 255).to(DEVICE) for _, path in batch]
        begin = time.time()
        with torch.autocast(DEVICE, enabled=DEVICE == 'cuda'):
            outputs = model(images)
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        seconds += time.time() - begin
        for (image_id, _), output in zip(batch, outputs):
            boxes = output['boxes'].float().cpu().numpy().astype(float)
            boxes[:, 2:] -= boxes[:, :2]
            parts.append(np.column_stack([np.full(len(boxes), image_id), boxes, output['scores'].float().cpu().numpy(),
                                          output['labels'].cpu().numpy()]))
    return np.concatenate(parts), 1000 * seconds / len(frames)


def run_torchvision(name, section, settings, make_model, **train_args):
    TORCHVISION_BUILDERS[name] = make_model
    if not should_run(name):
        return
    folder = Path(RUNS) / name
    weights = folder / 'model.pt'
    if not weights.exists() and not TRAIN:
        print('Not trained yet:', name)
        return
    folder.mkdir(parents=True, exist_ok=True)
    model = make_model()
    if weights.exists():
        model.load_state_dict(torch.load(weights, map_location='cpu'))
        state = json.loads((folder / 'train_state.json').read_text())
    else:
        state = train_torchvision(model, folder, **train_args)
        torch.save(model.state_dict(), f'{weights}.part')
        os.replace(f'{weights}.part', weights)
        (folder / 'train_state.json').write_text(json.dumps(state))
        (folder / 'checkpoint.pt').unlink(missing_ok=True)
    rows, ms = torchvision_rows(model)
    finish(name, section, settings, rows, ms, state['minutes'],
           spec=json.dumps(dict(kind='torchvision', run=name, weights=name)))
    del model
    torch.cuda.empty_cache()

### Anchor sizes from the data
Anchors must match box sizes after the model resizes the frame, so the protocol train boxes are rescaled the way the detector does it. The default smallest anchor is 32 px.

In [ ]:
sides = []
for path in tqdm(train_frames):
    width, height = Image.open(path).size
    labels = read_labels(path)
    scale = min(MIN_SIDE / min(width, height), IMGSZ / max(width, height))
    sides.append(np.sqrt(labels[:, 3] * width * labels[:, 4] * height) * scale)
side = pd.Series(np.concatenate(sides))

print(side.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(1))
for limit in (8, 16, 32):
    print(f'boxes under {limit} px after resizing: {(side < limit).mean():.1%}')

SMALL_ANCHORS = (16, 32, 64, 128, 256)

## Two-stage anchor-based: Faster R-CNN
`fasterrcnn_resnet50_fpn_v2` from torchvision: pretrained COCO weights, SGD with momentum 0.9, linear warmup and a cosine schedule, batch 4, mixed precision.

In [ ]:
run_torchvision('frcnn-base', 'two-stage', 'default anchors', build_faster_rcnn)

### Experiment 1: small anchors
Anchors from 16 px instead of 32 px, following the size table above.

In [ ]:
run_torchvision('frcnn-anchors', 'two-stage', 'small anchors', lambda: build_faster_rcnn(anchors=SMALL_ANCHORS))

### Experiment 2: GIoU box loss
Small anchors, and GIoU loss instead of smooth L1 for the second-stage box regression.

In [ ]:
run_torchvision('frcnn-giou', 'two-stage', 'small anchors, GIoU box loss',
                lambda: build_faster_rcnn(anchors=SMALL_ANCHORS, box_loss='giou'))

### Failure cases
Frames where the best Faster R-CNN misses the most objects, one frame per clip. A box is missed when no detection with score above 0.3 overlaps it with IoU 0.5, whatever the class. Missed objects are red, detections green.

In [ ]:
FAIL_RUN = best_run('two-stage')
SCORE = 0.3

rows = np.load(Path(RUNS) / FAIL_RUN / 'val_predictions.npy')
rows = rows[rows[:, 5] > SCORE]
paths = dict(val_frames)
truth = pd.DataFrame([a for a in val_coco['annotations'] if not a['iscrowd'] and a['image_id'] in paths])


def xyxy(boxes):
    boxes = torch.tensor(np.asarray(boxes, dtype=float).reshape(-1, 4))
    return torch.cat([boxes[:, :2], boxes[:, :2] + boxes[:, 2:]], dim=1)


misses = []
for image_id, objects in truth.groupby('image_id'):
    found = rows[rows[:, 0] == image_id, 1:5]
    best = box_iou(xyxy(objects.bbox.tolist()), xyxy(found)).max(1).values if len(found) else torch.zeros(len(objects))
    missed = objects[(best < 0.5).numpy()]
    misses.append({'image_id': image_id, 'clip': str(paths[image_id].parent.name), 'objects': len(objects),
                   'missed': len(missed), 'missed_median_side': np.sqrt(missed.area).median(), 'boxes': missed.bbox.tolist()})
misses = pd.DataFrame(misses)
worst = misses.sort_values('missed', ascending=False).drop_duplicates('clip').head(3)
display(worst.drop(columns='boxes').round(1))

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, frame in zip(axes, worst.itertuples()):
    image = np.array(Image.open(paths[frame.image_id]).convert('RGB'))
    ax.imshow(image)
    for x, y, w, h in rows[rows[:, 0] == frame.image_id, 1:5]:
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, color='#1baf7a', lw=0.8))
    for x, y, w, h in frame.boxes:
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, color='red', lw=1.2))
    ax.set_title(f'{frame.clip}: {frame.missed} of {frame.objects} missed', loc='left')
    ax.axis('off')
plt.tight_layout()
plt.show()

## One-stage anchor-based: RetinaNet
`retinanet_resnet50_fpn_v2` from torchvision under the same protocol and training loop.

In [ ]:
run_torchvision('retinanet-base', 'one-stage anchor', 'default anchors', build_retinanet)

### Experiment 1: small anchors

In [ ]:
run_torchvision('retinanet-anchors', 'one-stage anchor', 'small anchors', lambda: build_retinanet(anchors=SMALL_ANCHORS))

### Experiment 2: focal loss gamma 1.5
Small anchors, and focal loss gamma 1.5 instead of 2.0, so easy examples are down-weighted less.

In [ ]:
run_torchvision('retinanet-focal', 'one-stage anchor', 'small anchors, focal gamma 1.5',
                lambda: build_retinanet(anchors=SMALL_ANCHORS, gamma=1.5))

### Compared with the two-stage detector

In [ ]:
def compare(runs, title):
    log = load_log().set_index('run')
    rows = log.loc[[r for r in runs if r in log.index]]
    display(rows[['section', 'settings', 'mAP', 'mAP_testlike', 'AP50', 'AP_small', 'AP_medium', 'AP_large', 'ms_per_frame']].round(3))
    ax = rows[['AP_small', 'AP_medium', 'AP_large']].plot.barh(figsize=(9, 0.55 * len(rows) + 1.5), width=0.8,
                                                              color=['#2a78d6', '#eb6834', '#1baf7a'])
    ax.set_title(title, loc='left')
    ax.invert_yaxis()
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(['small', 'medium', 'large'], frameon=False, loc='lower right')
    plt.tight_layout()
    plt.show()


compare(['retinanet-base', 'retinanet-anchors', 'retinanet-focal', best_run('two-stage')],
        'AP by object size: RetinaNet vs the best Faster R-CNN')

## One-stage anchor-free: YOLO
YOLO11n from Ultralytics under the same protocol.

In [ ]:
run_ultralytics('yolo11n-base', 'anchor-free', 'default', lambda: YOLO('yolo11n.pt'))

### Experiment 1: input size 1280
At 1280 px, 9.8% of boxes are under 16 px, compared with 24.0% at 960.

In [ ]:
run_ultralytics('yolo11n-1280', 'anchor-free', 'imgsz 1280', lambda: YOLO('yolo11n.pt'), imgsz=1280, batch=4)

### Experiment 2: box loss gain 10
The box regression loss weight goes from the default 7.5 to 10.

In [ ]:
run_ultralytics('yolo11n-box', 'anchor-free', 'box loss gain 10', lambda: YOLO('yolo11n.pt'), box=10.0)

### Compared with the anchor-based detectors, by object size

In [ ]:
compare(['yolo11n-base', 'yolo11n-1280', 'yolo11n-box', best_run('two-stage'), best_run('one-stage anchor')],
        'AP by object size: YOLO vs the best anchor-based detectors')

## Query-based: RT-DETR
RT-DETR-L from Ultralytics under the same protocol, with AdamW.

In [ ]:
run_ultralytics('rtdetr-base', 'query-based', 'lr0 1e-4', lambda: RTDETR('rtdetr-l.pt'),
                batch=4, lr0=1e-4, optimizer='AdamW')

### Experiment 1: learning rate 2e-4

In [ ]:
run_ultralytics('rtdetr-lr', 'query-based', 'lr0 2e-4', lambda: RTDETR('rtdetr-l.pt'),
                batch=4, lr0=2e-4, optimizer='AdamW')

### Experiment 2: input size 640

In [ ]:
run_ultralytics('rtdetr-640', 'query-based', 'imgsz 640, lr0 1e-4', lambda: RTDETR('rtdetr-l.pt'),
                imgsz=640, batch=8, lr0=1e-4, optimizer='AdamW')

### Experiment 3: longer training (convergence check)
All RT-DETR curves still rise at the last epoch, so 4 epochs do not show what the model can do. The 640 px run is repeated with 16 epochs, 4 times the budget. This checks convergence; it is not part of the equal-budget comparison.

In [ ]:
run_ultralytics('rtdetr-640-long', 'query-based', 'imgsz 640, lr0 1e-4, 16 epochs', lambda: RTDETR('rtdetr-l.pt'),
                imgsz=640, epochs=16, batch=8, lr0=1e-4, optimizer='AdamW')

### Val mAP over training
RT-DETR against YOLO11n on the same frames and epochs, from each run's own log. Ultralytics computes these values with its own evaluator, so they are close to the table, not equal.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name, color in [('yolo11n-base', '#2a78d6'), ('rtdetr-base', '#eb6834'), ('rtdetr-lr', '#1baf7a'),
                    ('rtdetr-640', '#8e44ad'), ('rtdetr-640-long', '#555555')]:
    log_file = Path(RUNS) / name / 'results.csv'
    if log_file.exists():
        log = pd.read_csv(log_file)
        ax.plot(log['epoch'], log['metrics/mAP50-95(B)'], marker='o', color=color, label=name)
ax.set_xlabel('epoch')
ax.set_ylabel('val mAP50-95')
ax.set_title('Val mAP over training, same frames and epochs', loc='left')
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## All detectors compared

In [ ]:
table = load_log()
table = table[table.section != 'pipeline'].sort_values('mAP', ascending=False)
display(table[['run', 'section', 'settings', 'mAP', 'mAP_testlike', 'AP50', 'AP_small', 'ms_per_frame', 'train_minutes']].round(3))

order = table.sort_values('mAP')
fig, axes = plt.subplots(1, 3, figsize=(16, 0.42 * len(order) + 1.4), sharey=True)
for ax, col, title in zip(axes, ['mAP', 'AP_small', 'ms_per_frame'], ['mAP', 'AP_small', 'Inference, ms per frame']):
    ax.barh(order.run, order[col], color='#2a78d6')
    for y, value in enumerate(order[col]):
        ax.text(value, y, f' {value:.3f}' if col != 'ms_per_frame' else f' {value:.0f}', va='center', fontsize=9)
    ax.set_title(title, loc='left')
    ax.set_xticks([])
    ax.margins(x=0.18)
    ax.spines[['top', 'right', 'bottom']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
per_class = table.set_index('run')[[f'AP_{c}' for c in CLASSES]]
fig, ax = plt.subplots(figsize=(10, 0.42 * len(per_class) + 1.4))
ax.imshow(per_class.values, cmap='Blues', aspect='auto', vmin=0)
for (y, x), value in np.ndenumerate(per_class.values):
    ax.text(x, y, f'{value:.2f}', ha='center', va='center', fontsize=8,
            color='white' if value > per_class.values.max() * 0.6 else 'black')
ax.set_xticks(range(len(CLASSES)), CLASSES)
ax.set_yticks(range(len(per_class)), per_class.index)
ax.set_title('AP per class on val', loc='left')
plt.tight_layout()
plt.show()

## Your Pipeline
Starts from the leaderboard baseline: YOLO11n on all prepared frames (every 5th frame), 20 epochs, `imgsz=960`, public score 0.232. The data experiment changes only the training frames, experiments 1-2 change only inference, experiments 3-4 change only the model and experiment 5 changes only the checkpoint. Experiments 6 and 8 change the model and experiment 7 the training data (augmentation). The inference experiments then change only prediction settings on fixed weights, and the ensemble and temporal steps work on saved predictions, so the effects stay separate.

The final model is the best row of this section by `SELECT_METRIC`. Rows with `best.pt` are optimistic: their epoch was chosen on the same val frames that score them.

### Data experiment: training frame step
YOLO11n on every 5th, 10th, 15th and 20th frame with the same budget of about 9,700 training images, so a smaller step means more different frames, each seen fewer times. Warmup lasts the same number of iterations in every run.

In [ ]:
STEP_EPOCHS = {5: 2, 10: 4, 15: 6, 20: 8}
PREPARED_STEP = 5
prepared = sorted((PREP / 'train' / 'images').rglob('*.jpg'))

for step, epochs in STEP_EPOCHS.items():
    frames = prepared[::step // PREPARED_STEP]
    (PREP / f'train_step{step}.txt').write_text(''.join(f'{p}\n' for p in frames))
    step_yaml = PREP / f'step{step}.yaml'
    step_yaml.write_text(f'path: {PREP}\ntrain: train_step{step}.txt\nval: val_protocol.txt\n'
                         f'nc: {len(CLASSES)}\nnames: {json.dumps(CLASSES)}\n')
    run_ultralytics(f'yolo11n-step{step}', 'frame step', f'every {step}th frame, {len(frames)} frames x {epochs} epochs',
                    lambda: YOLO('yolo11n.pt'), data=step_yaml, epochs=epochs, warmup_epochs=epochs / 8)

In [ ]:
steps = load_log().query("section == 'frame step'").copy()
steps['step'] = steps.run.str.extract(r'step(\d+)', expand=False).astype(int)
steps = steps.sort_values('step')
display(steps[['run', 'settings', 'mAP', 'AP50', 'AP_small', 'train_minutes']].round(3))

fig, ax = plt.subplots(figsize=(7, 4))
for col, color in [('mAP', '#2a78d6'), ('AP_small', '#eb6834')]:
    ax.plot(steps.step, steps[col], marker='o', color=color, label=col)
    for x, y in zip(steps.step, steps[col]):
        ax.annotate(f'{y:.3f}', (x, y), textcoords='offset points', xytext=(0, 7), ha='center', fontsize=9)
ax.set_xticks(steps.step)
ax.set_xlabel('training frame step')
ax.set_title('Val AP by frame step, same training budget', loc='left')
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

### Training recipes
The full-data recipes are kept in one place, so the final model can be retrained on train + val with exactly the same settings.

In [ ]:
FULL_DATA = PREP / 'data.yaml'
RECIPES = {
    'baseline-yolo11n-960': dict(make_model=lambda: YOLO('yolo11n.pt'), epochs=20, batch=-1, warmup_epochs=3.0),
    'yolo11n-1280-full': dict(make_model=lambda: YOLO('yolo11n.pt'), imgsz=1280, epochs=20, batch=-1,
                              warmup_epochs=3.0),
    'yolo11s-960-full': dict(make_model=lambda: YOLO('yolo11s.pt'), epochs=20, batch=-1, warmup_epochs=3.0),
    'yolo11s-1280': dict(make_model=lambda: YOLO('yolo11s.pt'), imgsz=1280, epochs=20, batch=-1, warmup_epochs=3.0),
    'yolo11s-1280-aug': dict(make_model=lambda: YOLO('yolo11s.pt'), imgsz=1280, epochs=20, batch=-1,
                             warmup_epochs=3.0, scale=0.9, mixup=0.1, cos_lr=True),
    'yolov8s-p2-1280': dict(make_model=lambda: YOLO('yolov8s-p2.yaml').load('yolov8s.pt'), imgsz=1280, epochs=20,
                            batch=-1, warmup_epochs=3.0),
}

In [ ]:
run_ultralytics('baseline-yolo11n-960', 'pipeline', 'baseline', data=FULL_DATA,
                **RECIPES['baseline-yolo11n-960'])

### Experiment 1: inference at 1280
The baseline weights, with frames resized to 1280 instead of 960 at prediction time.

In [ ]:
run_ultralytics('pipe-infer-1280', 'pipeline', 'baseline weights, predict at 1280', weights='baseline-yolo11n-960', imgsz=1280)

### Experiment 2: test-time augmentation
Ultralytics `augment=True`: the frame is predicted at three scales, one of them flipped, and the boxes are merged before NMS.

In [ ]:
run_ultralytics('pipe-tta', 'pipeline', 'baseline weights, test-time augmentation', weights='baseline-yolo11n-960', augment=True)

### Experiment 3: training at 1280
The baseline recipe with `imgsz=1280`.

In [ ]:
run_ultralytics('yolo11n-1280-full', 'pipeline', 'yolo11n, imgsz 1280, 20 epochs', data=FULL_DATA,
                **RECIPES['yolo11n-1280-full'])

### Experiment 4: larger model
The baseline recipe with YOLO11s instead of YOLO11n.

In [ ]:
run_ultralytics('yolo11s-960-full', 'pipeline', 'yolo11s, imgsz 960, 20 epochs', data=FULL_DATA,
                **RECIPES['yolo11s-960-full'])

### Experiment 5: checkpoint picked on val
The two full trainings with `best.pt` (highest val mAP during training) instead of the last epoch.

In [ ]:
for weights, imgsz in [('yolo11n-1280-full', 1280), ('yolo11s-960-full', 960)]:
    run_ultralytics(f'{weights}-best', 'pipeline', f'{weights}, best.pt', weights=weights, imgsz=imgsz,
                    checkpoint='best')


### Experiment 6: YOLO11s at 1280 (model)
Experiment 3 showed that 1280 px helps YOLO11n, and experiment 4 tried YOLO11s only at 960. This run combines both: YOLO11s with the experiment 3 recipe. It is scored with `last.pt`, so the number is not picked on val.

In [ ]:
run_ultralytics('yolo11s-1280', 'pipeline', 'yolo11s, imgsz 1280, 20 epochs', data=FULL_DATA, **RECIPES['yolo11s-1280'])

### Experiment 7: stronger augmentation (data)
YOLO11s at 960 had its best val mAP at epoch 3 and then went down, so it overfits the 56 train clips. Same model and input size as experiment 6, with scale jitter 0.9 instead of 0.5, mixup 0.1 (Zhang et al., Bag of Freebies) and a cosine learning rate.

In [ ]:
run_ultralytics('yolo11s-1280-aug', 'pipeline', 'yolo11s, imgsz 1280, scale 0.9, mixup 0.1, cosine lr',
                data=FULL_DATA, **RECIPES['yolo11s-1280-aug'])

### Experiment 8 (optional): extra high-resolution head (model)
The smallest YOLO head has stride 8, so an object under 16 px covers at most a few cells of it, and 9.8% of boxes are still under 16 px at 1280. YOLOv8s with an extra stride-4 head (`yolov8s-p2.yaml`, the rest initialised from `yolov8s.pt`) gives these objects more locations. It is the slowest run, so it only runs with `TRY_P2_HEAD`.

In [ ]:
if TRY_P2_HEAD:
    run_ultralytics('yolov8s-p2-1280', 'pipeline', 'yolov8s with a P2 head, imgsz 1280, 20 epochs', data=FULL_DATA,
                    **RECIPES['yolov8s-p2-1280'])

### Inference experiments (data and inference strategy)
The best trained YOLO stays fixed and only the prediction settings change, one at a time. Each step keeps the best value by `SELECT_METRIC` and passes it on:
- score threshold 0.01 or 0.001: mAP uses the whole ranking, and up to 500 boxes per frame are scored;
- NMS IoU 0.5, 0.6 or 0.7: scenes are dense and 43-65% of boxes are occluded, so neighbours overlap;
- input size 1280, 1536 or 1920: no test frame is wider than 1920 px;
- tiles: the frame is also cut into 2x2 tiles with 20% overlap, each predicted at the same input size, and all boxes are merged with class-wise NMS, as in SAHI (Akyon et al., 2022). Frames whose long side is not above the input size are not tiled;
- test-time augmentation.

In [ ]:
def sweep(base, key, values, label):
    candidates = []
    for value in values:
        spec = {**base, key: value}
        spec['run'] = base['run'] if value == base[key] else inference_name(spec)
        if spec['run'] != base['run']:
            run_spec(spec, f"{base['weights']}, {label} {value}")
        candidates.append(spec)
    log = load_log().set_index('run')
    done = [spec for spec in candidates if spec['run'] in log.index]
    return max(done, key=lambda spec: log.loc[spec['run'], SELECT_METRIC]) if done else base


TRAINED = ['yolo11s-1280-aug', 'yolo11s-1280', 'yolov8s-p2-1280', 'yolo11n-1280-full']
log = load_log().set_index('run')
INFER_BASE = max([r for r in TRAINED if r in log.index], key=lambda r: log.loc[r, SELECT_METRIC])
print('Weights for the inference experiments:', INFER_BASE)

spec = spec_of(INFER_BASE)
spec = sweep(spec, 'conf', [SCORE_THRESHOLD, 0.001], 'score threshold')
spec = sweep(spec, 'iou', [0.5, 0.6, NMS_IOU], 'NMS IoU')
spec = sweep(spec, 'imgsz', sorted({spec['imgsz'], 1536, 1920}), 'input size')
spec = sweep(spec, 'tiles', [0, 2], 'tiles')
spec = sweep(spec, 'augment', [False, True], 'test-time augmentation')
print('Selected:', {k: spec[k] for k in ('imgsz', 'conf', 'iou', 'tiles', 'augment')})

steps = load_log()[lambda d: d.run.str.startswith(f'{INFER_BASE}-')]
display(steps[['run', 'settings', 'mAP', SELECT_METRIC, 'AP_small', 'ms_per_frame']].round(3))

### Faster R-CNN on all prepared frames (model)
Faster R-CNN was the best required detector (val mAP 0.201) after only 1,623 frames x 4 epochs with a 960 px long side. Here it gets all 4,868 prepared frames, 6 epochs and the torchvision default size (short side 800, long side up to 1333), with the small anchors. The batch is 2 to fit a T4, and the learning rate is halved with it. It is trained as a second, different model for the ensemble below.

In [ ]:
run_torchvision('frcnn-full', 'pipeline', 'small anchors, 800-1333 px, all prepared frames, 6 epochs',
                lambda: build_faster_rcnn(anchors=SMALL_ANCHORS, min_size=800, max_size=1333),
                frames=sorted((PREP / 'train' / 'images').rglob('*.jpg')), epochs=6, batch=2, lr=LR / 2)

### Ensemble with Weighted Boxes Fusion (model)
YOLO and Faster R-CNN make different mistakes. Weighted Boxes Fusion (Solovyev et al., 2019; the `ensemble-boxes` library) averages boxes that overlap across models instead of keeping only one of them, and lowers the score of a box that only one model found. It runs on saved predictions, so it needs no GPU on val.

In [ ]:
def wbf_rows(member_rows, split, weights, iou_thr):
    groups = [by_image(rows) for rows in member_rows]
    parts = []
    for image_id in tqdm(sorted(set().union(*groups)), leave=False):
        width, height = SIZES[split][image_id]
        scale = np.array([width, height, width, height], dtype=float)
        boxes, scores, labels = [], [], []
        for group in groups:
            rows = group.get(image_id, np.zeros((0, 7)))
            boxes.append((np.column_stack([rows[:, 1:3], rows[:, 1:3] + rows[:, 3:5]]) / scale).clip(0, 1))
            scores.append(rows[:, 5])
            labels.append(rows[:, 6])
        fused, fused_scores, fused_labels = weighted_boxes_fusion(boxes, scores, labels, weights=weights,
                                                                  iou_thr=iou_thr, skip_box_thr=0.0)
        keep = np.argsort(-fused_scores)[:MAX_DET]
        parts.append(to_rows(image_id, fused[keep] * scale, fused_scores[keep], fused_labels[keep] - 1))
    return np.concatenate(parts)


log = load_log()
singles = [r for r in log.query("section == 'pipeline'").dropna(subset=[SELECT_METRIC])
           .sort_values(SELECT_METRIC, ascending=False).run if spec_of(r)['kind'] == 'ultralytics']
if singles and 'frcnn-full' in set(log.run):
    single = singles[0]
    for weights, iou_thr in [([1, 1], 0.55), ([2, 1], 0.55), ([2, 1], 0.65)]:
        run_spec(dict(kind='ensemble', run=f'wbf-{single}+frcnn-full-w{weights[0]}{weights[1]}-iou{iou_thr}',
                      members=[spec_of(single), spec_of('frcnn-full')], weights=weights, iou_thr=iou_thr),
                 f'WBF of {single} and frcnn-full, weights {weights}, IoU {iou_thr}')
else:
    print('Needs frcnn-full and a scored YOLO first')

### Bonus: temporal rescoring
Val and test frames are consecutive. A real object is usually found again at almost the same place in the next frame, while many false positives are not. Each box is matched to the box of the same class with the highest IoU in each of the `k` previous and next frames (IoU at least 0.3; a neighbour without a match counts as score 0), and its score becomes `(1 - w) * score + w * mean neighbour score`. Boxes do not move, only the ranking changes. This is a simplified Seq-NMS (Han et al., 2016) without the linking step. The grid is small (k = 1 or 2, w = 0.3 or 0.6) because it is tuned on the same val clips that score it; `paired` below shows on how many clips it helps.

In [ ]:
def boxes_xyxy(rows):
    return torch.tensor(np.column_stack([rows[:, 1:3], rows[:, 1:3] + rows[:, 3:5]]))


def temporal_rows(rows, split, window, weight, iou_thr):
    groups = by_image(rows)
    frames = pd.DataFrame([(i, *ORDER[split][i]) for i in groups], columns=['image_id', 'clip', 'frame'])
    parts = []
    for _, clip in frames.groupby('clip'):
        lookup = dict(zip(clip.frame, clip.image_id))
        for image_id, frame in zip(clip.image_id, clip.frame):
            current = groups[image_id]
            boxes, labels = boxes_xyxy(current), torch.tensor(current[:, 6])
            total, count = torch.zeros(len(current), dtype=torch.float64), 0
            for offset in [k for k in range(-window, window + 1) if k]:
                other = groups.get(lookup.get(frame + offset))
                if other is None:
                    continue
                iou = box_iou(boxes, boxes_xyxy(other))
                iou[labels[:, None] != torch.tensor(other[:, 6])[None]] = 0
                best = iou.max(1)
                total += torch.where(best.values >= iou_thr, torch.tensor(other[:, 5])[best.indices], 0.0)
                count += 1
            rescored = current.copy()
            if count:
                rescored[:, 5] = (1 - weight) * current[:, 5] + weight * total.numpy() / count
            parts.append(rescored)
    return np.concatenate(parts)


log = load_log().query("section == 'pipeline'").dropna(subset=[SELECT_METRIC])
bases = [r for r in log.sort_values(SELECT_METRIC, ascending=False).run
         if spec_of(r)['kind'] in ('ultralytics', 'ensemble')]
base = bases[0]
names = []
for window, weight in [(1, 0.3), (1, 0.6), (2, 0.3), (2, 0.6)]:
    names.append(f'{base}+temporal-k{window}-w{weight}')
    run_spec(dict(kind='temporal', run=names[-1], base=spec_of(base), window=window, weight=weight, iou_thr=0.3),
             f'temporal rescoring of {base}, k={window}, w={weight}')
display(paired([name for name in names if name in set(load_log().run)], base))

### All steps compared
Val mAP on all clips and on the test-like clips, then every run against the baseline clip by clip: the mean gain over the 7 clips and on how many of them the run is better.

In [ ]:
pipeline = load_log().query("section == 'pipeline'").sort_values(SELECT_METRIC)
display(pipeline[['run', 'settings', 'mAP', SELECT_METRIC, 'AP50', 'AP_small', 'ms_per_frame']].round(3))

fig, axes = plt.subplots(1, 2, figsize=(16, 0.42 * len(pipeline) + 1.4), sharey=True)
for ax, col in zip(axes, ['mAP', SELECT_METRIC]):
    ax.barh(pipeline.run, pipeline[col], color='#2a78d6')
    for y, value in enumerate(pipeline[col]):
        ax.text(value, y, f' {value:.3f}', va='center', fontsize=9)
    ax.set_title(f'Your Pipeline: val {col}', loc='left')
    ax.set_xticks([])
    ax.margins(x=0.15)
    ax.spines[['top', 'right', 'bottom']].set_visible(False)
plt.tight_layout()
plt.show()

display(paired(pipeline.run[::-1], 'baseline-yolo11n-960'))

## Final model and submission
The best row of Your Pipeline by `SELECT_METRIC` decides the models and the inference settings. Its full description (`spec`) is stored in the log, so the test frames are predicted exactly as the val frames were.

In [ ]:
final = load_log().query("section == 'pipeline'").dropna(subset=[SELECT_METRIC]).nlargest(1, SELECT_METRIC).iloc[0]
FINAL_SPEC = spec_of(final.run)
print(f'Final: {final.run} ({final.settings})')
print(f'val mAP {final.mAP:.3f}, {SELECT_METRIC} {final[SELECT_METRIC]:.3f}')

### Retraining on train + val
With `USE_TRAINVAL`, every YOLO in the final model is retrained with its recipe on train plus every 5th val frame. Val adds 7 clips to the 56 in train, including the only 4K footage. No held-out frames are left, so this model uses `last.pt` after the fixed number of epochs and is not scored on val. Both the original and the retrained version are submitted, so one of each can be chosen as a final Kaggle submission.

In [ ]:
def model_specs(spec):
    if spec['kind'] in MODEL_KINDS:
        yield spec
    for child in spec.get('members', []) + ([spec['base']] if 'base' in spec else []):
        yield from model_specs(child)


def with_trainval(spec):
    original, spec = json.dumps(spec, sort_keys=True), dict(spec)
    if spec['kind'] == 'ultralytics' and spec['weights'] in TRAINVAL:
        spec.update(weights=TRAINVAL[spec['weights']], checkpoint='last')
    if 'members' in spec:
        spec['members'] = [with_trainval(member) for member in spec['members']]
    if 'base' in spec:
        spec['base'] = with_trainval(spec['base'])
    if json.dumps(spec, sort_keys=True) != original:
        spec['run'] += '-trainval'
    return spec


trainval_frames = (sorted((PREP / 'train' / 'images').rglob('*.jpg'))
                   + sorted((PREP / 'val' / 'images').rglob('*.jpg'))[::TRAIN_STRIDE])
(PREP / 'train_trainval.txt').write_text(''.join(f'{p}\n' for p in trainval_frames))
trainval_yaml = PREP / 'trainval.yaml'
trainval_yaml.write_text(f'path: {PREP}\ntrain: train_trainval.txt\nval: val_protocol.txt\n'
                         f'nc: {len(CLASSES)}\nnames: {json.dumps(CLASSES)}\n')

TRAINVAL = {}
for member in model_specs(FINAL_SPEC) if USE_TRAINVAL else []:
    weights = member['weights']
    if member['kind'] != 'ultralytics' or weights not in RECIPES:
        continue
    run_ultralytics(f'{weights}-trainval', 'final', f'{weights} recipe on train + val', data=trainval_yaml,
                    evaluate=False, val=False, **RECIPES[weights])
    last = Path(RUNS) / f'{weights}-trainval' / 'weights' / 'last.pt'
    if last.exists() and not unfinished(last):
        TRAINVAL[weights] = f'{weights}-trainval'
print(f'{len(trainval_frames)} frames; retrained:', TRAINVAL or 'none')

### Submission file
One row per test image in `test_images.csv`. `PredictionString` is a list of `class score x y width height` groups, with (x, y) as the top-left corner in pixels of the original image. Images with no detections stay empty, and at most 500 detections per image are kept, highest score first.

Test predictions of every model are saved on Drive as `test_predictions.npy`, so an interrupted run continues where it stopped and the ensemble and temporal steps reuse them.

In [ ]:
def to_submission(rows):
    sizes = test_images[['width', 'height']].to_numpy(float)
    strings = [''] * len(test_images)
    for image_id, part in by_image(rows).items():
        part = part[np.argsort(-part[:, 5], kind='stable')][:MAX_DET]
        width, height = sizes[image_id]
        x1, y1 = part[:, 1].clip(0, width), part[:, 2].clip(0, height)
        x2, y2 = (part[:, 1] + part[:, 3]).clip(0, width), (part[:, 2] + part[:, 4]).clip(0, height)
        groups = zip(part[:, 6].astype(int) - 1, part[:, 5], x1, y1, x2 - x1, y2 - y1)
        strings[image_id] = ' '.join(f'{c} {s:.5f} {x:.1f} {y:.1f} {w:.1f} {h:.1f}' for c, s, x, y, w, h in groups)
    return pd.DataFrame({'image_id': test_images.image_id, 'PredictionString': strings})


SUBMISSIONS = {FINAL_SPEC['run']: FINAL_SPEC}
if TRAINVAL:
    trainval_spec = with_trainval(FINAL_SPEC)
    SUBMISSIONS[trainval_spec['run']] = trainval_spec

pending = [s['run'] for spec in SUBMISSIONS.values() for s in model_specs(spec)
           if not (Path(RUNS) / s['run'] / 'test_predictions.npy').exists()]
assert not pending or torch.cuda.is_available(), f'{pending} still need a GPU runtime'

files = {name: to_submission(cached_rows(spec, 'test')) for name, spec in SUBMISSIONS.items()}
MAIN = list(files)[-1]
submission = files[MAIN]
print('Main submission:', MAIN)
submission.head()

Check the file before using a submission: every test image appears once, each detection has 6 values, classes are 0-7, and boxes stay inside their frame.

In [ ]:
def check_submission(submission):
    values = submission.PredictionString.fillna('').str.split()
    counts = values.str.len() // 6
    assert len(submission) == len(test_images) and submission.image_id.is_unique
    assert (values.str.len() % 6 == 0).all() and counts.max() <= MAX_DET

    detections = pd.DataFrame(
        [[i, *group] for i, row in zip(submission.image_id, values)
         for group in np.array(row, dtype=float).reshape(-1, 6)],
        columns=['image_id', 'cls', 'score', 'x', 'y', 'w', 'h']).merge(test_images, on='image_id')

    assert detections.cls.between(0, len(CLASSES) - 1).all() and detections.score.between(0, 1).all()
    assert (detections.w >= 0).all() and (detections.h >= 0).all()
    assert (detections.x >= -1).all() and (detections.y >= -1).all()
    assert (detections.x + detections.w <= detections.width + 1).all()
    assert (detections.y + detections.h <= detections.height + 1).all()
    return detections, counts


checked = {name: check_submission(file) for name, file in files.items()}
for name, (found, counts) in checked.items():
    print(f'{name}: {len(found)} detections, {(counts == 0).sum()} empty images')
detections = checked[MAIN][0]
print(detections.cls.astype(int).map(lambda i: CLASSES[i]).value_counts())

In [ ]:
samples = test_images.drop_duplicates(['width', 'height']).sort_values('width').iloc[[0, -1]]
sure = detections[detections.score > 0.3]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, frame in zip(axes, samples.itertuples()):
    image = cv2.cvtColor(cv2.imread(str(DATA / frame.file_name)), cv2.COLOR_BGR2RGB)
    draw_boxes(image, sure[sure.image_id == frame.image_id], (0, 220, 0))
    ax.imshow(image)
    ax.set_title(f'{frame.clip} ({frame.width}x{frame.height}), score > 0.3')
    ax.axis('off')
plt.tight_layout()
plt.show()

The boxes above should sit on objects in both the smallest and the largest frame size. If they are shifted or scaled, the coordinates are wrong, so do not submit.

In [ ]:
for name, file in files.items():
    file.to_csv(Path(RUNS) / f'submission-{name}.csv', index=False)
submission.to_csv('submission.csv', index=False)
print(Path('submission.csv').stat().st_size / 1e6, 'MB')

In [ ]:
if SUBMIT and not SMOKE_TEST:
    for name in files:
        file = Path(RUNS) / f'submission-{name}.csv'
        !kaggle competitions submit -c {COMPETITION} -f "{file}" -m "{name}"
    !kaggle competitions submissions -c {COMPETITION}

In [ ]:
stop_runtime()